<a href="https://colab.research.google.com/github/leodavidfan/AI_Books/blob/main/CNN_accuracy_simply_Praxis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CNN classifier for defect vs normal images, train the model with training folder, test the model with testing folder
ChatGPT said:

Perfect — you want a CNN classifier for defect vs normal, with separate training and testing folders. I’ll give you a complete PyTorch script that:

Loads training data from train/

Loads testing data from test/

Trains a simple CNN

Evaluates accuracy on the test set

Folder structure

dataset/
   train/
      normal/
      defect/
   test/
      normal/
      defect/

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
data_dir = "/content/drive/MyDrive/casting_512x512/casting_512x512/"
data_dir512_def = "/content/drive/MyDrive/casting_512x512/casting_512x512/def_front"
data_dir512_ok = "/content/drive/MyDrive/casting_512x512/casting_512x512/ok_front"
data_dir256_train = "/content/drive/MyDrive/casting_data/train"
data_dir256_test = "/content/drive/MyDrive/casting_data/test"

data_dir256_train_ok = "/content/drive/MyDrive/casting_data/train/ok_front"

# add generated iamges into the folder
gendata_dir = "/content/drive/MyDrive/generated_images_ConDiff_090125/"

# add 300*300 images into the folder w training and rename the subfolders to normal and defect
data_dir_training = "/content/drive/MyDrive/training"
data_dir_testing = "/content/drive/MyDrive/testing"

# add generated images
data_dir_gen1 = "/content/drive/MyDrive/generated_images_ConDiff_090625_2modified"

# add combined images (training + generated)
data_dir_com1 = "/content/drive/MyDrive/combined_images_ConDiff_090625_2modified"


In [2]:
# new dataset for chap 3 & 4
data_dir_testing = "/content/drive/MyDrive/Praxis_Testing_Dataset/casting_data_origin/test"
data_dir_training = "/content/drive/MyDrive/Praxis_Testing_Dataset/casting_data_origin/train"

data_dir_testing_add100 = "/content/drive/MyDrive/Praxis_Testing_Dataset/casting_data_add100/test"
data_dir_training_add100 = "/content/drive/MyDrive/Praxis_Testing_Dataset/casting_data_add100/train"

data_dir_testing_add200 = "/content/drive/MyDrive/Praxis_Testing_Dataset/casting_data_add200/test"
data_dir_training_add200 = "/content/drive/MyDrive/Praxis_Testing_Dataset/casting_data_add200/train"


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# -------------------
# CONFIG
# -------------------
DATASET_PATH = "dataset"
IMAGE_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 15
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------
# DATASETS
# -------------------
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])  # normalize to [-1,1]
])

# train_dataset = datasets.ImageFolder(f"{DATASET_PATH}/train", transform=transform)
# test_dataset  = datasets.ImageFolder(f"{DATASET_PATH}/test",  transform=transform)
train_dataset = datasets.ImageFolder(data_dir_training, transform=transform) # real images
# train_dataset = datasets.ImageFolder(data_dir_com1, transform=transform) # combined images
test_dataset  = datasets.ImageFolder(data_dir_testing,  transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print("Classes:", train_dataset.classes)  # ['defect', 'normal'] or vice versa

# -------------------
# CNN MODEL
# -------------------
class DefectCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(DefectCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # -> 64x64

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # -> 32x32

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # -> 16x16
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * (IMAGE_SIZE // 8) * (IMAGE_SIZE // 8), 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = DefectCNN(num_classes=2).to(DEVICE)

# -------------------
# TRAINING
# -------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    acc = 100. * correct / total
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {running_loss/len(train_loader):.4f}, Train Acc: {acc:.2f}%")

# -------------------
# TESTING
# -------------------
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

test_acc = 100. * correct / total
print(f"Test Accuracy: {test_acc:.2f}%")

# -------------------
# SAVE MODEL
# -------------------
torch.save(model.state_dict(), "cnn_defect_vs_normal.pt")


Classes: ['defect', 'normal']
Epoch 1/15, Loss: 0.4127, Train Acc: 80.75%
Epoch 2/15, Loss: 0.2544, Train Acc: 89.67%
Epoch 3/15, Loss: 0.1852, Train Acc: 92.72%
Epoch 4/15, Loss: 0.1285, Train Acc: 95.13%
Epoch 5/15, Loss: 0.0888, Train Acc: 97.32%
Epoch 6/15, Loss: 0.0618, Train Acc: 98.15%
Epoch 7/15, Loss: 0.0559, Train Acc: 98.16%
Epoch 8/15, Loss: 0.0461, Train Acc: 98.60%
Epoch 9/15, Loss: 0.0359, Train Acc: 98.84%
Epoch 10/15, Loss: 0.0318, Train Acc: 99.20%
Epoch 11/15, Loss: 0.0269, Train Acc: 99.25%
Epoch 12/15, Loss: 0.0242, Train Acc: 99.31%
Epoch 13/15, Loss: 0.0231, Train Acc: 99.37%
Epoch 14/15, Loss: 0.0250, Train Acc: 99.16%
Epoch 15/15, Loss: 0.0186, Train Acc: 99.50%
Test Accuracy: 99.44%
